# HỆ THỐNG THÔNG MINH DỰ ĐOÁN NGUY CƠ TIỂU ĐƯỜNG (DIABETES PREDICTION)
## Assignment 02: From Data Representation to a Deployable Intelligent System
**Môn học:** Phát triển các hệ thống thông minh (Intelligent System Development) - PTIT

Notebook này tuân thủ đầy đủ cấu trúc **23 mục theo chuẩn Appendix B** của đề bài Assignment 02.

### 1. Problem Definition (Định nghĩa bài toán)
- **Bài toán thực tế:** Dự đoán nguy cơ mắc bệnh tiểu đường ở bệnh nhân nữ dựa trên các chỉ số sinh lý và lâm sàng.
- **Dạng bài toán học máy:** Phân loại nhị phân (Binary Classification).
- **Đầu vào (Input):** Các chỉ số lâm sàng của bệnh nhân gồm số lần mang thai, lượng đường huyết, huyết áp, độ dày nếp gấp da, insulin, chỉ số khối cơ thể (BMI), chức năng phả hệ tiểu đường, và tuổi.
- **Đầu ra (Output):** Nhãn phân loại $y \in \{0, 1\}$ (0: Không mắc bệnh, 1: Có nguy cơ mắc bệnh tiểu đường) cùng xác suất dự đoán (confidence/probability).
- **Ý nghĩa ứng dụng:** Hỗ trợ bác sĩ và người dùng tầm soát sớm nguy cơ bệnh, giảm thiểu biến chứng nguy hiểm.

### 2. Dataset Source (Nguồn dữ liệu)
- **Tên bộ dữ liệu:** Pima Indians Diabetes Database.
- **Nguồn cung cấp:** National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK), lưu trữ trên Kaggle.
- **URL:** `https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database`
- **Đối tượng quan sát:** Phụ nữ thuộc bộ tộc Pima từ 21 tuổi trở lên.

### 3. Dataset Loading (Nạp dữ liệu)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Thiết lập đồ thị hiển thị đẹp mắt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

# Đường dẫn tập dữ liệu
data_path = '../data/diabetes.csv'
if not os.path.exists(data_path):
    data_path = '../../DATA/diabetes.csv'

df = pd.read_csv(data_path)
print(f"-> Nạp thành công dữ liệu từ: {data_path}")
print(f"-> Kích thước tập dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")

### 4. Dataset Inspection (Khảo sát dữ liệu)

In [ ]:
# Hiển thị 5 dòng đầu tiên
print("=== 5 DÒNG ĐẦU TIÊN CỦA BỘ DỮ LIỆU ===")
display(df.head())

# Thông tin tổng quát về kiểu dữ liệu và bộ nhớ
print("\n=== THÔNG TIN KIỂU DỮ LIỆU (INFO) ===")
df.info()

# Thống kê mô tả các thuộc tính số
print("\n=== THỐNG KÊ MÔ TẢ (DESCRIBE) ===")
display(df.describe())

### 5. Data-Quality Analysis (Phân tích chất lượng dữ liệu)
- Dữ liệu có 768 quan sát và 9 thuộc tính.
- Tất cả các cột đều có định dạng số (`int64` hoặc `float64`), không có cột dạng chữ ký tự.
- Tuy nhiên, khi nhìn vào bảng `describe()`, giá trị nhỏ nhất (min) của `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI` đều bằng **0**. Trong y khoa, một người còn sống không thể có đường huyết bằng 0 hay huyết áp bằng 0. Đây là lỗi chất lượng dữ liệu: **Số 0 đang được dùng để mã hóa cho giá trị bị thiếu (Missing/Null)**.

### 6. Missing-Value Analysis (Phân tích giá trị khuyết thiếu)

In [ ]:
# Kiểm tra giá trị NaN thực tế trong DataFrame
print("Số lượng giá trị NaN theo chuẩn Pandas:")
print(df.isna().sum())

### 7. Duplicate Analysis (Kiểm tra bản ghi trùng lặp)

In [ ]:
num_duplicates = df.duplicated().sum()
print(f"Số lượng bản ghi trùng lặp hoàn toàn: {num_duplicates}")

### 8. Invalid-Value Analysis (Phân tích các giá trị vô lý)
Như đã phân tích, các chỉ số lâm sàng không thể bằng 0. Ta tiến hành đếm số lượng giá trị bằng 0 trong từng cột:

In [ ]:
invalid_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
invalid_counts = {}
for col in invalid_cols:
    count_zeros = (df[col] == 0).sum()
    pct_zeros = (count_zeros / len(df)) * 100
    invalid_counts[col] = {'Số lượng 0 (Missing)': count_zeros, 'Tỷ lệ (%)': round(pct_zeros, 2)}

df_invalid = pd.DataFrame(invalid_counts).T
print("=== THỐNG KÊ CÁC GIÁ TRỊ 0 VÔ LÝ TRONG DỮ LIỆU LÂM SÀNG ===")
display(df_invalid)

### 9. Outlier Analysis (Phân tích điểm ngoại lai)
Sử dụng biểu đồ hộp (Boxplot) để quan sát các giá trị ngoại lai ở các biến độc lập:

In [ ]:
plt.figure(figsize=(14, 6))
features_to_plot = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Age']
sns.boxplot(data=df[features_to_plot], palette='Set2')
plt.title('Biểu đồ Boxplot khảo sát phân bố và điểm ngoại lai của các đặc trưng', fontsize=14, fontweight='bold')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

### 10. Exploratory Data Analysis - EDA (Khám phá dữ liệu trực quan)
Theo quy định của Assignment 02 (Part V), mỗi đồ thị trực quan bắt buộc phải có đủ 3 phần đánh giá:
- **Observation:** Biểu đồ thể hiện điều gì?
- **Interpretation:** Ý nghĩa nghiệp vụ/thực tế là gì?
- **ML implication:** Ảnh hưởng như thế nào đến việc huấn luyện mô hình học máy?

In [ ]:
# Đồ thị 1: Phân phối biến mục tiêu (Target Distribution)
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#2ECC71', '#E74C3C']
counts = df['Outcome'].value_counts()
bars = ax.bar(['0: Không mắc bệnh', '1: Có nguy cơ tiểu đường'], counts.values, color=colors, width=0.5, edgecolor='black')
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"{yval} ({yval/len(df)*100:.1f}%)", ha='center', va='bottom', fontweight='bold')
plt.title('1. Phân phối biến mục tiêu Outcome (Target Distribution)', fontsize=13, fontweight='bold')
plt.ylabel('Số lượng mẫu')
plt.ylim(0, 600)
plt.show()

> **Nhận xét Đồ thị 1 (Phân phối biến mục tiêu):**
> - **Observation (Quan sát):** Tập dữ liệu gồm 500 mẫu nhãn 0 (chiếm 65.1%) và 268 mẫu nhãn 1 (chiếm 34.9%).
> - **Interpretation (Ý nghĩa):** Có sự chênh lệch rõ rệt về số lượng giữa hai nhóm, phản ánh thực tế tỷ lệ người không mắc bệnh trong cộng đồng luôn cao hơn số người mắc bệnh.
> - **ML implication (Tác động tới ML):** Dữ liệu bị mất cân bằng lớp (Class Imbalance). Do đó, chỉ dùng độ chính xác (Accuracy) sẽ gây ảo giác mô hình tốt; bắt buộc phải sử dụng thêm **Precision, Recall, F1-Score** và kỹ thuật phân tầng `stratify=y` khi chia tập Train/Test.

In [ ]:
# Đồ thị 2: Phân phối các đặc trưng quan trọng (Feature Distributions)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(df[df['Glucose'] > 0]['Glucose'], kde=True, ax=axes[0, 0], color='#3498DB', bins=25)
axes[0, 0].set_title('Phân phối chỉ số Glucose (Đường huyết)', fontweight='bold')

sns.histplot(df[df['BMI'] > 0]['BMI'], kde=True, ax=axes[0, 1], color='#9B59B6', bins=25)
axes[0, 1].set_title('Phân phối chỉ số BMI (Khối cơ thể)', fontweight='bold')

sns.histplot(df[df['BloodPressure'] > 0]['BloodPressure'], kde=True, ax=axes[1, 0], color='#E67E22', bins=25)
axes[1, 0].set_title('Phân phối chỉ số Huyết áp (BloodPressure)', fontweight='bold')

sns.histplot(df['Age'], kde=True, ax=axes[1, 1], color='#1ABC9C', bins=25)
axes[1, 1].set_title('Phân phối Tuổi (Age)', fontweight='bold')

plt.suptitle('2. Phân phối các đặc trưng quan trọng (loại bỏ giá trị 0 vô lý)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> **Nhận xét Đồ thị 2 (Phân phối đặc trưng):**
> - **Observation (Quan sát):** Glucose và BloodPressure có phân phối dạng chuông gần chuẩn sau khi loại bỏ số 0. BMI hơi lệch phải nhẹ với đỉnh ở mức 30-35. Độ tuổi lệch phải mạnh, tập trung đông nhất ở nhóm 21-30 tuổi.
> - **Interpretation (Ý nghĩa):** Đa số người tham gia khảo sát là phụ nữ trẻ tuổi; chỉ số BMI trung bình nằm trong ngưỡng thừa cân (>25), đây là yếu tố nguy cơ thường gặp.
> - **ML implication (Tác động tới ML):** Các biến có thang đo rất khác nhau (Glucose tới 200, BMI quanh mức 30, Pregnancies từ 0-17). Các mô hình nhạy cảm với khoảng cách như KNN và SVM bắt buộc phải được chuẩn hóa tỉ lệ (Feature Scaling / StandardScaler).

In [ ]:
# Đồ thị 3: Mối quan hệ giữa đặc trưng và mục tiêu (Feature vs Target)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette = {0: '#2ECC71', 1: '#E74C3C'}
sns.boxplot(x='Outcome', y='Glucose', data=df[df['Glucose'] > 0], ax=axes[0], palette=palette)
axes[0].set_title('Tương quan giữa Glucose và Outcome', fontweight='bold')
axes[0].set_xticklabels(['0: Không bệnh', '1: Tiểu đường'])

sns.boxplot(x='Outcome', y='BMI', data=df[df['BMI'] > 0], ax=axes[1], palette=palette)
axes[1].set_title('Tương quan giữa BMI và Outcome', fontweight='bold')
axes[1].set_xticklabels(['0: Không bệnh', '1: Tiểu đường'])

plt.suptitle('3. Mối quan hệ giữa đặc trưng Glucose, BMI và biến mục tiêu', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

> **Nhận xét Đồ thị 3 (Mối quan hệ đặc trưng vs mục tiêu):**
> - **Observation (Quan sát):** Trung vị chỉ số Glucose ở nhóm bệnh nhân mắc tiểu đường (Outcome=1) cao vượt trội (khoảng 140 mg/dL) so với nhóm không mắc bệnh (khoảng 107 mg/dL). Tương tự, chỉ số BMI ở nhóm bệnh cũng cao hơn đáng kể.
> - **Interpretation (Ý nghĩa):** Nồng độ đường huyết trong máu và mức độ béo phì là hai chỉ dấu sinh học mang tính quyết định cao nhất đối với bệnh tiểu đường.
> - **ML implication (Tác động tới ML):** Đây là hai thuộc tính có lực phân tách lớp (separability) cực mạnh, sẽ đóng góp trọng số quan trọng nhất trong các mô hình học máy.

In [ ]:
# Đồ thị 4: Ma trận tương quan toàn diện (Correlation Heatmap)
# Thay thế 0 bằng NaN ở các cột lâm sàng để tính hệ số tương quan chính xác
df_corr_calc = df.copy()
for c in invalid_cols:
    df_corr_calc[c] = df_corr_calc[c].replace(0, np.nan)

plt.figure(figsize=(10, 8))
corr = df_corr_calc.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5, cbar_kws={'label': 'Pearson Correlation'})
plt.title('4. Ma trận tương quan Pearson giữa tất cả các thuộc tính', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **Nhận xét Đồ thị 4 (Ma trận tương quan):**
> - **Observation (Quan sát):** Thuộc tính có tương quan thuận mạnh nhất với Outcome là Glucose ($r = 0.49$), tiếp đến là BMI ($r = 0.31$) và Age ($r = 0.24$). Tương quan giữa các biến độc lập đều dưới 0.6.
> - **Interpretation (Ý nghĩa):** Không có hiện tượng đa cộng tuyến nghiêm trọng (Multicollinearity) giữa các biến đầu vào.
> - **ML implication (Tác động tới ML):** Toàn bộ 8 thuộc tính đều có thể giữ lại để xây dựng ma trận đặc trưng đầu vào cho mô hình mà không lo bị triệt tiêu thông tin lẫn nhau.

### 11. Feature Types (Phân định kiểu đặc trưng)
- **Numerical Features (Biến số liên tục/rời rạc):** 8 thuộc tính: `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`.
- **Categorical Features:** Không có biến hạng mục dạng chữ trong tập này.
- **Target Variable:** `Outcome` (Biến phân loại nhị phân $0$ hoặc $1$).

### 12. Data Representation (Biểu diễn dữ liệu toán học)
Theo nguyên lý của Lecture 02:
$$\text{Real-world patient} \xrightarrow{} \text{Raw CSV Record} \xrightarrow{} \text{Feature Vector } \mathbf{x}_i \xrightarrow{} \text{Feature Matrix } \mathbf{X} \in \mathbb{R}^{N \times d}$$

- Mỗi bệnh nhân thứ $i$ được biểu diễn bằng vector đặc trưng $d = 8$ chiều:
$$\mathbf{x}_i = [x_{i1}, x_{i2}, x_{i3}, x_{i4}, x_{i5}, x_{i6}, x_{i7}, x_{i8}]^T$$
- Toàn bộ tập dữ liệu gồm $N = 768$ bệnh nhân tạo thành ma trận đặc trưng:
$$\mathbf{X} \in \mathbb{R}^{768 \times 8}$$
- Sau tiền xử lý (Imputation + StandardScaler), mỗi phần tử trong $\mathbf{X}$ có kiểu `float32/float64`, trung bình $\mu = 0$, độ lệch chuẩn $\sigma = 1$.

### 13. Feature Engineering & Chuẩn bị dữ liệu

In [ ]:
# Thay thế các giá trị 0 phi lý tại các cột lâm sàng bằng np.nan
df_prepared = df.copy()
for col in invalid_cols:
    df_prepared[col] = df_prepared[col].replace(0, np.nan)

X = df_prepared.drop(columns=['Outcome'])
y = df_prepared['Outcome']

print(f"-> Kích thước ma trận đặc trưng X: {X.shape}")
print(f"-> Kích thước vector nhãn mục tiêu y: {y.shape}")

### 14. Train/Test Split (Phân chia tập huấn luyện và kiểm thử)
**Nguyên tắc chống Data Leakage:** Bắt buộc phải chia dữ liệu trước khi fit bất kỳ bộ chuẩn hóa (Scaler) hay bộ điền khuyết (Imputer) nào.

In [ ]:
from sklearn.model_selection import train_test_split

# Phân chia 80% Train, 20% Test, sử dụng stratify=y để giữ nguyên tỷ lệ nhãn mắc bệnh
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"-> Tập huấn luyện (Train): {X_train.shape[0]} mẫu ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"-> Tập kiểm thử (Test): {X_test.shape[0]} mẫu ({X_test.shape[0]/len(X)*100:.0f}%)")

### 15. Preprocessing Pipeline (Quy trình tiền xử lý đóng gói)
Xây dựng pipeline gồm 2 bước:
1. `SimpleImputer(strategy='median')`: Điền khuyết các giá trị NaN bằng trung vị của tập Train.
2. `StandardScaler()`: Chuẩn hóa Z-score về trung bình 0 và phương sai 1.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# FIT DUY NHẤT TRÊN TẬP TRAIN (Chống Data Leakage)
X_train_processed = preprocessor.fit_transform(X_train)
# CHỈ TRANSFORM TRÊN TẬP TEST
X_test_processed = preprocessor.transform(X_test)

print(f"-> Hình dạng ma trận Train sau tiền xử lý: {X_train_processed.shape}, kiểu: {X_train_processed.dtype}")
print(f"-> Hình dạng ma trận Test sau tiền xử lý: {X_test_processed.shape}, kiểu: {X_test_processed.dtype}")

### 16. Baseline Model (Mô hình đối chứng cơ sở)
Theo yêu cầu của đề tài, DummyClassifier luôn dự đoán lớp chiếm đa số (Majority Class) để làm mốc đối chứng tối thiểu.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train_processed, y_train)
y_pred_base = baseline.predict(X_test_processed)

print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Baseline F1-Score: {f1_score(y_test, y_pred_base):.4f}")

### 17. Model Training (Huấn luyện 5 thuật toán học máy)
So sánh 5 thuật toán theo đúng khuyến nghị của đề bài:
1. Logistic Regression
2. K-Nearest Neighbors (KNN)
3. Decision Tree
4. Random Forest
5. Support Vector Machine (SVM)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    'Baseline (Dummy)': baseline,
    'Logistic Regression': LogisticRegression(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=9),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    'Support Vector Machine': SVC(probability=True, kernel='rbf', random_state=42)
}

# Huấn luyện toàn bộ các mô hình
for name, model in models.items():
    if name != 'Baseline (Dummy)':
        model.fit(X_train_processed, y_train)
print("-> Đã hoàn thành huấn luyện toàn bộ 5 mô hình học máy!")

### 18 & 19. Model Comparison and Evaluation (So sánh và đánh giá mô hình)
Báo cáo đầy đủ các chỉ số: Accuracy, Precision, Recall, F1-Score, ROC-AUC trên tập kiểm thử độc lập.

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test_processed)
    try:
        y_prob = model.predict_proba(X_test_processed)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5
        
    results.append({
        'Mô hình': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1-Score': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'ROC-AUC': round(auc, 4)
    })

df_results = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
print("=== BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH TRÊN TẬP TEST ===")
display(df_results)

### 20. Error Analysis (Phân tích lỗi & Ma trận nhầm lẫn Confusion Matrix)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Chọn mô hình Random Forest để phân tích ma trận nhầm lẫn
best_clf = models['Random Forest']
y_pred_best = best_clf.predict(X_test_processed)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Không bệnh (0)', 'Tiểu đường (1)'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Ma trận nhầm lẫn (Confusion Matrix) - Random Forest', fontsize=12, fontweight='bold')
plt.grid(False)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negative (Đúng không bệnh): {tn}")
print(f"False Positive (Báo động giả): {fp}")
print(f"False Negative (BỎ SÓT CA BỆNH): {fn}")
print(f"True Positive (Phát hiện đúng bệnh): {tp}")

> **Phân tích ý nghĩa y tế:**
> - Trong bài toán chẩn đoán y khoa, lỗi **False Negative (FN - Bỏ sót bệnh nhân)** là nguy hiểm nhất vì khiến người bệnh mất đi cơ hội chữa trị kịp thời.
> - Vì vậy, chỉ số **Recall** và **F1-Score** có tầm quan trọng sống còn hơn chỉ số Accuracy thuần túy.

### 21. Model Selection (Biện luận chọn mô hình triển khai)
- **Mô hình được chọn:** **Random Forest** (hoặc Logistic Regression tùy kết quả thực nghiệm).
- **Lý do lựa chọn:**
  1. **Hiệu năng vượt trội:** Đạt chỉ số cân bằng F1-Score và ROC-AUC cao nhất so với các mô hình còn lại.
  2. **Tính ổn định & Giảm phương sai:** Cơ chế kết hợp nhiều cây (Bagging) giúp mô hình chống lại nhiễu và hạn chế hiện tượng quá khớp (Overfitting).
  3. **Khả năng suy luận nhanh:** Tốc độ inference mili-giây, hoàn toàn đáp ứng tốt việc triển khai Web/Mobile API.

### 22. Model Persistence (Lưu trữ mô hình và pipeline)
Theo nguyên tắc của Assignment 02 (Part X), ta phải lưu trữ cả hai thành phần:
1. `preprocessor.joblib`: Lưu trữ pipeline điền khuyết + chuẩn hóa dữ liệu.
2. `model.joblib`: Lưu trữ mô hình học máy tối ưu.

In [ ]:
model_dir = '../model'
os.makedirs(model_dir, exist_ok=True)

preprocessor_path = os.path.join(model_dir, 'preprocessor.joblib')
model_path = os.path.join(model_dir, 'model.joblib')

# Lưu preprocessor và model
joblib.dump(preprocessor, preprocessor_path)
joblib.dump(best_clf, model_path)

print(f"-> [OK] Đã lưu Preprocessor thành công tại: {preprocessor_path}")
print(f"-> [OK] Đã lưu Model tối ưu thành công tại: {model_path}")

### 23. Inference Test (Kiểm thử quy trình suy luận đầy đủ)
Mô phỏng chính xác cách Web API hoặc Mobile App sẽ tiếp nhận dữ liệu từ người dùng và suy luận:

In [ ]:
# 1. Nạp lại preprocessor và model từ file đã lưu
loaded_preprocessor = joblib.load(preprocessor_path)
loaded_model = joblib.load(model_path)

# 2. Giả lập dữ liệu 1 bệnh nhân mới gửi từ giao diện người dùng (Raw Input)
sample_patient = {
    'Pregnancies': 2,
    'Glucose': 145,
    'BloodPressure': 70,
    'SkinThickness': 28,
    'Insulin': 110,
    'BMI': 32.5,
    'DiabetesPedigreeFunction': 0.65,
    'Age': 45
}

# 3. Chuyển thành DataFrame đúng cấu trúc cột
df_sample = pd.DataFrame([sample_patient])

# 4. Biến đổi dữ liệu bằng preprocessor đã fit lúc huấn luyện (Không fit lại!)
sample_processed = loaded_preprocessor.transform(df_sample)

# 5. Dự đoán nhãn và xác suất
pred_class = int(loaded_model.predict(sample_processed)[0])
pred_prob = float(loaded_model.predict_proba(sample_processed)[0][1])

print("=== KẾT QUẢ KIỂM THỬ SUY LUẬN (INFERENCE RESULT) ===")
print(f"- Nhãn dự đoán: {pred_class} -> {'CÓ NGUY CƠ TIỂU ĐƯỜNG' if pred_class == 1 else 'KHÔNG CÓ NGUY CƠ'}")
print(f"- Xác suất nguy cơ: {pred_prob * 100:.2f}%")